# Stroke Prediction – Logistic Regression

This notebook loads the stroke prediction dataset, preprocesses the data, trains a logistic regression model, evaluates performance, and analyzes feature importance (including BMI).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report
)

from IPython.display import display

## 1. Load and Inspect the Dataset

In [ ]:
# Load the dataset
# Try the common filenames used in this project
import os
candidates = [
    "Stroke_Prediction_Dataset.xlsx",
    "Stroke_Prediction_Dataset_41868569.xlsx",
    "/home/workdir/attachments/Stroke_Prediction_Dataset.xlsx",
]
df = None
for path in candidates:
    if os.path.exists(path):
        df = pd.read_excel(path)
        print(f"Loaded: {path}")
        break
if df is None:
    raise FileNotFoundError("Could not find the Stroke Prediction Excel file. "
                            "Place it in the same folder as this notebook.")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())
print("\nDiagnosis distribution:")
print(df["Diagnosis"].value_counts())


# ============================================================
# 2. Prepare Target Variable and Features  (memory-safe version)
# ============================================================

target_column = "Diagnosis"

# Drop rows that have no diagnosis
df = df.dropna(subset=[target_column]).copy()

# Binary target: Stroke = 1, No Stroke = 0
diagnosis = (
    df[target_column]
    .astype(str)
    .str.strip()
    .str.casefold()
)
y = diagnosis.map({"stroke": 1, "no stroke": 0})

if y.isna().any():
    unknown = diagnosis[y.isna()].unique()
    raise ValueError(f"Unexpected diagnosis values: {unknown}")

# ------------------------------------------------------------------
# CRITICAL: drop any column that is an identifier or has too many
# unique values (especially Patient Name).  One-hot encoding a name
# column creates ~15 000 extra columns and causes the memory error.
# ------------------------------------------------------------------
drop_exact = {
    target_column,
    "ID", "Patient ID", "Patient_ID", "PatientID",
    "Patient Name", "patient name", "Name", "name"
}

# Also drop any object/string column whose number of unique values
# is more than 5 % of the number of rows (high-cardinality)
high_card_threshold = max(50, int(0.05 * len(df)))

cols_to_drop = []
for col in df.columns:
    if col in drop_exact:
        cols_to_drop.append(col)
        continue
    if df[col].dtype == object or str(df[col].dtype).startswith("string"):
        nunique = df[col].nunique(dropna=True)
        if nunique > high_card_threshold:
            print(f"Dropping high-cardinality column: {col} ({nunique} unique values)")
            cols_to_drop.append(col)

X = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()
print("Columns kept for modelling:", list(X.columns))
print("Shape before encoding:", X.shape)

# Convert any remaining boolean columns to string so they become categorical
for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype(str)

# Identify categorical columns that still need one-hot encoding
categorical_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("\nCategorical columns that will be one-hot encoded:")
print(categorical_columns)

# Fill missing values
for col in categorical_columns:
    X[col] = X[col].fillna("Unknown").astype(str)

numeric_columns = X.select_dtypes(include="number").columns.tolist()
for col in numeric_columns:
    X[col] = X[col].fillna(X[col].median())

# One-hot encode (now safe – only low-cardinality categoricals remain)
X = pd.get_dummies(X, columns=categorical_columns, dtype="int8")  # int8 saves memory

# Final numeric matrix
X = X.astype("float32")   # float32 uses half the RAM of float64

print("\nFinal feature matrix shape:", X.shape)
print("(If this is still > a few hundred columns, something is wrong)")
display(X.head())


In [ ]:
# Identify the target column
target_column = "Diagnosis"

# Remove rows without a diagnosis
df = df.dropna(subset=[target_column]).copy()

# Convert diagnosis into binary values
# Stroke = 1, No Stroke = 0
diagnosis = (
    df[target_column]
    .astype(str)
    .str.strip()
    .str.casefold()
)

y = diagnosis.map({
    "stroke": 1,
    "no stroke": 0
})

if y.isna().any():
    unknown = diagnosis[y.isna()].unique()
    raise ValueError(f"Unexpected diagnosis values: {unknown}")

# Remove the target, identifier, and name columns from the predictors
# IMPORTANT: Patient Name must be excluded – otherwise one-hot encoding
# creates ~15,000 columns and causes memory errors.
exclude_columns = [
    target_column,
    "ID",
    "Patient ID",
    "Patient_ID",
    "PatientID",
    "Patient Name"          # <-- critical fix
]

X = df.drop(
    columns=[c for c in exclude_columns if c in df.columns]
).copy()

# Convert Boolean columns into categorical columns
for column in X.select_dtypes(include="bool").columns:
    X[column] = X[column].astype(str)

# Identify categorical columns
categorical_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Categorical columns that will be one-hot encoded:")
print(categorical_columns)

# Fill missing categorical values
for column in categorical_columns:
    X[column] = X[column].fillna("Unknown").astype(str)

# Fill missing numerical values with their medians
numeric_columns = X.select_dtypes(
    include="number"
).columns.tolist()

for column in numeric_columns:
    X[column] = X[column].fillna(X[column].median())

# One-hot encode categorical variables
X = pd.get_dummies(
    X,
    columns=categorical_columns,
    dtype=int
)

# Ensure all model inputs are numeric
X = X.astype(float)

print("\nNumber of features after encoding:", X.shape[1])
print("Encoded dataset shape:", X.shape)
display(X.head())


## 3. Train/Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

# Standardize the features using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data split and scaling completed.")

## 4. Train Logistic Regression Model

In [ ]:
# Create the logistic regression model
model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train the model
model.fit(X_train_scaled, y_train)

print("Logistic regression model trained successfully.")

## 5. Evaluate Model Performance

In [ ]:
# Predict stroke diagnoses
y_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r_squared = r2_score(y_test, y_pred)

print("MODEL PERFORMANCE")
print("-----------------")
print(f"Accuracy:  {accuracy:.4f} ({accuracy:.2%})")
print(f"Precision: {precision:.4f} ({precision:.2%})")
print(f"Recall:    {recall:.4f} ({recall:.2%})")
print(f"MAE:       {mae:.4f}")
print(f"MSE:       {mse:.4f}")
print(f"R-squared: {r_squared:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Stroke", "Stroke"],
    zero_division=0
))

## 6. Feature Importance

In [ ]:
# Extract model coefficients
coefficients = model.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients,
    "Absolute Importance": np.abs(coefficients)
})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="Absolute Importance",
    ascending=False
).reset_index(drop=True)

print("TOP 15 FEATURES")
display(feature_importance.head(15))

# Show the strongest positive and negative predictors
print("\nStrongest positive predictors:")
display(
    feature_importance.sort_values(
        by="Coefficient",
        ascending=False
    ).head(10)
)

print("\nStrongest negative predictors:")
display(
    feature_importance.sort_values(
        by="Coefficient",
        ascending=True
    ).head(10)
)

In [ ]:
# Plot the top 15 features
top_features = feature_importance.head(15).sort_values(
    by="Coefficient"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Coefficient"]
)

plt.title("jaybat1453 - Logistic Regression Feature Importance")
plt.xlabel("Standardized Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.axvline(0, linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. BMI Analysis

In [ ]:
# Find the BMI column
bmi_column = "Body Mass Index (BMI)"

if bmi_column in df.columns:
    bmi_values = pd.to_numeric(
        df[bmi_column],
        errors="coerce"
    ).dropna()

    bmi_25 = bmi_values.quantile(0.25)
    bmi_75 = bmi_values.quantile(0.75)

    # Find the corresponding encoded feature
    bmi_index = list(X.columns).index(bmi_column)

    # Use one testing record as the baseline
    baseline = X_test.iloc[[0]].copy()

    # Compare predictions at the 25th and 75th BMI percentiles
    lower_bmi = baseline.copy()
    higher_bmi = baseline.copy()

    lower_bmi.iloc[0, bmi_index] = bmi_25
    higher_bmi.iloc[0, bmi_index] = bmi_75

    # Scale using the previously fitted scaler
    lower_scaled = scaler.transform(lower_bmi)
    higher_scaled = scaler.transform(higher_bmi)

    lower_probability = model.predict_proba(
        lower_scaled
    )[0, 1]

    higher_probability = model.predict_proba(
        higher_scaled
    )[0, 1]

    bmi_coefficient = model.coef_[0][bmi_index]

    print("BMI ANALYSIS")
    print("------------")
    print(f"25th percentile BMI: {bmi_25:.2f}")
    print(f"75th percentile BMI: {bmi_75:.2f}")
    print(f"BMI coefficient: {bmi_coefficient:.4f}")
    print(
        f"Predicted stroke probability at lower BMI: "
        f"{lower_probability:.2%}"
    )
    print(
        f"Predicted stroke probability at higher BMI: "
        f"{higher_probability:.2%}"
    )
    print(
        f"Probability difference: "
        f"{higher_probability - lower_probability:+.2%}"
    )

else:
    print("The BMI column was not found. Check the dataset column names.")